In [1]:
import instructor
from enum import Enum
from openai import AzureOpenAI
from functools import partial
import pandas as pd
from datetime import datetime
from typing import Optional, List, TypeVar, Type
from pydantic import BaseModel, Field, model_validator, ValidationInfo, ConfigDict
import re
import os
from Entity_response_model import test

In [2]:
# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = 'YOUR-API-KEY'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    temperature=0,
    max_retries=2  # Retries in case of API issues
)

In [3]:
input_dir = '../input'
df = pd.DataFrame(columns=['File Name', 'Inclusion Criteria', 'Exclusion Criteria'])
abstract_list = [] 

for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            ### Split the notes to inclusion and exclusion criteria seperately
            inclusion_criteria = content.split('Exclusion Criteria:')[0].split('Inclusion Criteria:')[-1].strip()
            exclusion_criteria = content.split('Exclusion Criteria:')[-1].strip()
            new_row = pd.DataFrame({'File Name': [file_name], 'Inclusion Criteria': [inclusion_criteria], 'Exclusion Criteria': [exclusion_criteria]})
            df = pd.concat([df, new_row], ignore_index=True)

abstract_list = df[['File Name', 'Inclusion Criteria', 'Exclusion Criteria']].values.tolist()
print("Abstracts have been successfully read into a list.")

Abstracts have been successfully read into a list.


In [ ]:
file_name, inclusion_criteria, exclusion_criteria = abstract_list[1]  
clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
type = "inclusion"
criteria = clean_text_ec
verb = "excluded"


In [5]:
# Use the Azure OpenAI client to generate the response
prompt = f'''Extract information from the clinical trial eligibility document ({type} criteria). Here is the {type} criteria section {criteria}.
Please ensure you distinguish between 'CYP3A4' and 'CYP3A'. They are distinct, and an instance of 'CYP3A' should not be interpreted as 'CYP3A4'.
'''
response = in_client(
    response_model = test,
    messages=[
        {
            "role": "system",
            "content": "You are an assistant that extracts named entities from text, classifies and summarizes text.",
        },
        {"role": "user", "content": prompt},
    ],
)

In [6]:
response

test(diag_down_syndrome_yn=<Yes_no.YES: 'Yes'>, diag_down_syndrome=<Yes_no.YES: 'Yes'>)

In [7]:
# Use the Azure OpenAI client to generate the response
prompt = f'''Extract information from the clinical trial eligibility document ({type} criteria). Here is the {type} criteria section {criteria}.
Please ensure you distinguish between 'CYP3A4' and 'CYP3A'. They are distinct, and an instance of 'CYP3A' should not be interpreted as 'CYP3A4'.
All the questions are formed in two parts. Must answer both parts of the questions. The answer for the first part of the question must be "Yes", or "No".
Do the eligibility criteria mention a diagnosis of Down syndrome? If so, is a patient with a diagnosis of Down syndrome to be {verb}?
'''
response = in_client(
    # response_model = test,
    messages=[
        {
            "role": "system",
            "content": "You are an assistant that extracts named entities from text, classifies and summarizes text.",
        },
        {"role": "user", "content": prompt},
    ],
)

In [8]:
response.choices[0].message

ChatCompletionMessage(content='Yes, the eligibility criteria mention a diagnosis of Down syndrome. A patient with a diagnosis of Down syndrome is to be excluded.', refusal=None, role='assistant', function_call=None, tool_calls=None)